# AegisForge-AI Demonstration Notebook

Welcome to the **AegisForge-AI** production demonstration. This notebook showcases how our engineering tools securely evaluate mechanical integrity while strictly adhering to our .agents/rules/architecture.md standards.

### Features Demonstrated:
- **Strict Validation**: All inputs and outputs are validated using Pydantic schemas (schemas/inspection.py) before execution.
- **Cryptographic Auditing**: Every tool integrates AuditLedger (	ools/audit_trail.py) to cryptographically log all engineering calculations, enabling secure statutory audits.
- **Scalability**: The core logic is decoupled into robust Python classes that can be securely deployed in high-risk environments.

### Available Tools:
1. **Material Lookup Tool**: Queries the offline ASME Section II Part D Database for critical material properties like allowable stress and yield strength.
2. **API 579 FFS Tool**: Performs Level 1 Fitness-For-Service (FFS) assessments for localized thin areas (LTA).
3. **API 581 RBI Tool**: Calculates Risk-Based Inspection (RBI) intervals based on remaining life, process pressure, and fluid toxicity.

In [4]:
import sys
import os
# Add root directory to python path for absolute imports regardless of where Jupyter was launched
sys.path.append(r'c:\Users\DELL\Desktop\SIH')


## 1. Material Lookup Tool

**What it does (Easy Explanation):**
This tool acts like a digital engineering handbook. When building or evaluating pressure vessels, engineers must know exactly how much stress a specific metal can handle at a specific temperature. This tool fetches verified physical properties (like allowable stress and yield strength) from the ASME database.

**Example Scenario:**
We need to know the properties of 2.25Cr-1Mo steel when it is operating at a high temperature of 350.0 ?C. We pass these inputs to the tool, and it returns the precise allowable stress in MPa.

In [5]:
from schemas.inspection import MaterialLookupInput
from tools.material_lookup_tool import MaterialLookupTool

material_tool = MaterialLookupTool(use_audit_trail=True)
inp = MaterialLookupInput(material_grade="2.25Cr-1Mo", temperature_c=350.0)
result = material_tool.lookup_material(inp, caller_agent="notebook")
print("Result:", result.model_dump())


Result: {'status': 'SUCCESS', 'error': None, 'material': '2.25Cr-1Mo (SA-387 Gr 22)', 'temperature_c': 350.0, 'allowable_stress_mpa': 138.0, 'yield_strength_mpa': 310.0, 'tensile_strength_mpa': 515.0}


## 2. API 579 Fitness-For-Service (FFS) Tool

**What it does (Easy Explanation):**
Over time, metal walls degrade and get thinner (e.g., due to corrosion). When an inspector finds a "Localized Thin Area" (LTA), this tool uses the API 579 engineering standard to calculate if the equipment is still strong enough to operate safely. It calculates a "Remaining Strength Factor" (RSF). If the RSF is 0.90 or higher, the equipment is generally safe.

**Example Scenario:**
We have a vessel that requires a thickness of 138.57 mm. However, an inspector found a thin spot that is only 138.2 mm thick. We input the flaw dimensions, and the tool evaluates whether we can keep running the plant or if we need an emergency repair.

In [6]:
from schemas.inspection import Api579FfsInput
from tools.api_579_ffs_tool import Api579FfsTool

ffs_tool = Api579FfsTool(use_audit_trail=True)
ffs_inp = Api579FfsInput(
    t_actual_global=140.0, 
    t_min_lta=138.2, 
    t_req=138.57, 
    flaw_length_mm=50.0, 
    inside_radius_mm=1200.0
)
ffs_result = ffs_tool.evaluate_lta(ffs_inp, caller_agent="notebook")
print("FFS Result:", ffs_result.model_dump())


FFS Result: {'status': 'SUCCESS', 'error': None, 'rsf': 1.0, 'allowable_rsf': 0.9, 'is_acceptable': True, 'action': 'Acceptable for continued operation based on Level 1 RSF assessment.'}


## 3. API 581 Risk-Based Inspection (RBI) Tool

**What it does (Easy Explanation):**
Instead of inspecting equipment on a rigid, fixed schedule (like every 5 years), Risk-Based Inspection (RBI) calculates *exactly when* the next inspection should happen based on real risk. This tool evaluates the remaining life of the equipment, the operating pressure, and how toxic the internal fluid is, to recommend a smart inspection date.

**Example Scenario:**
We are evaluating a vessel with only 2.5 years of life left. It operates at a very high pressure (14.5 MPa) and contains High toxicity fluid. The tool dynamically assesses this high-risk combination and recommends a much shorter, safer inspection interval.

In [7]:
from schemas.inspection import RbiIntervalInput
from tools.risk_based_inspection_tool import RiskBasedInspectionTool

rbi_tool = RiskBasedInspectionTool(use_audit_trail=True)
rbi_inp = RbiIntervalInput(
    remaining_life_years=2.5, 
    design_pressure_mpa=14.5, 
    fluid_toxicity="High"
)
rbi_result = rbi_tool.calculate_rbi_interval(rbi_inp, caller_agent="notebook")
print("RBI Result:", rbi_result.model_dump())


RBI Result: {'status': 'SUCCESS', 'error': None, 'risk_category': 'MEDIUM_HIGH', 'recommended_interval_months': 12, 'statutory_verdict': 'ENHANCED_INSPECTION_REQUIRED'}
